# Extração ORCID — Produção Bibliográfica

Este notebook substitui a versão original (que lia um único ORCID iD fixo)
por um **loop sobre uma lista de pessoas**, cada uma com seu `id_lattes` e
seu `orcid_id` mapeados.

O objetivo é gerar dois DataFrames **no mesmo formato de saída do `analyse.ipynb`**:

- `df_artigos_periodico_orcid` → mesmo schema de `df_artigos_final` (artigos de periódico)
- `df_artigos_congresso_orcid` → mesmo schema de `df_artigos_congresso_final` (trabalhos de congresso/conferência)

Assim, depois é possível simplesmente fazer:

```python
df_periodicos_unificado = pd.concat([df_artigos_final, df_artigos_periodico_orcid, df_artigos_periodico_scopus], ignore_index=True)
df_congressos_unificado = pd.concat([df_artigos_congresso_final, df_artigos_congresso_orcid], ignore_index=True)
```

e subir para o banco (DuckDB) usando o mesmo `CREATE TABLE` / `INSERT` que o `analyse.ipynb` já define.


## 1. Instalação e Configuração

In [ ]:
%pip install python-dotenv orcid

In [1]:
import time
import pandas as pd
import numpy as np
import os
import orcid
from dotenv import load_dotenv

load_dotenv()

# Credenciais da API pública do ORCID (Developer Tools -> https://orcid.org/developer-tools)
CLIENT_ID = os.getenv('ORCID_CLIENT_ID')
CLIENT_SECRET = os.getenv('ORCID_CLIENT_SECRET')

# Inicializa a API Pública (sandbox=False -> dados reais de produção)
api = orcid.PublicAPI(CLIENT_ID, CLIENT_SECRET, sandbox=False)

# Token de acesso de leitura pública (necessário para ler perfis)
token = api.get_search_token_from_orcid()
print("Autenticação realizada com sucesso!")

Autenticação realizada com sucesso!


## 2. Lista de pessoas a extrair

Aqui entra a lista de pessoas do seu projeto. Cada pessoa precisa ter:

- `id_lattes`: a chave que une todas as fontes (Lattes, ORCID, Scopus) — é o que vai virar a FK no banco.
- `orcid_id`: o ORCID iD da pessoa (formato `0000-0002-4258-0424`), usado só para consultar a API.

Substitua a lista de exemplo abaixo pela sua lista real (pode vir de um CSV/Excel/JSON com `pd.read_csv`, etc).

In [2]:
# Exemplo de lista de pessoas. Troque por:
#   df_pessoas_lista = pd.read_csv('lista_pessoas.csv')  # colunas: id_lattes, orcid_id, scopus_author_id
lista_pessoas_orcid = [
    {'id_lattes': '0000000000000001', 'orcid_id': '0000-0003-0057-7670'},
    # {'id_lattes': '...', 'orcid_id': '...'},
]

print(f"Total de pessoas a processar: {len(lista_pessoas_orcid)}")

Total de pessoas a processar: 1


## 3. Extração dos trabalhos (Works) de cada pessoa

Para cada pessoa, lemos o registro público completo do ORCID e extraímos a lista
de `works` (trabalhos). Cada trabalho é classificado em **periódico** ou
**congresso/evento** com base no campo `type` retornado pela API ORCID, e
guardamos os mesmos campos que o `analyse.ipynb` espera (título, veículo/revista
ou evento, ano, DOI, autores).

In [3]:
# Tipos de trabalho do ORCID que consideramos "artigo de periódico"
# (https://info.orcid.org/documentation/integration-guide/orcid-work-types/)
TIPOS_PERIODICO = {
    'journal_article',
}

# Tipos de trabalho do ORCID que consideramos "trabalho de congresso/conferência"
TIPOS_CONGRESSO = {
    'conference_paper',
}


def extrair_autores(trabalho):
    """Extrai os nomes de autores/contribuidores de um work-summary do ORCID,
    juntando-os em uma única string separada por vírgula (mesmo padrão usado
    em df_bib_artigos['autores'] e df_bib_trab_congresso['autores'] no analyse.ipynb)."""
    contribuidores = (trabalho.get('contributors') or {}).get('contributor', [])
    nomes = []
    for contrib in contribuidores:
        credit_name = (contrib.get('credit-name') or {})
        nome = credit_name.get('value') if credit_name else None
        if nome:
            nomes.append(nome.strip())
    return ', '.join(nomes) if nomes else pd.NA


def extrair_doi(trabalho):
    ext_ids_container = trabalho.get('external-ids') or {}
    ext_ids = ext_ids_container.get('external-id', []) if ext_ids_container else []
    for ident in ext_ids:
        if ident.get('external-id-type') == 'doi':
            return ident.get('external-id-value')
    return pd.NA


lista_artigos_periodico = []
lista_artigos_congresso = []

for pessoa in lista_pessoas_orcid:
    id_lattes = pessoa['id_lattes']
    orcid_id = pessoa['orcid_id']

    print(f"Processando ORCID {orcid_id} (id_lattes={id_lattes})...")

    try:
        perfil = api.read_record_public(orcid_id, 'record', token)
    except Exception as exc:
        print(f"  -> Falha ao ler perfil de {orcid_id}: {exc}")
        continue
    
    
    atividades = perfil.get('activities-summary', {})
    grupos_trabalhos = atividades.get('works', {}).get('group', [])
    print("OI", atividades);

    for grupo in grupos_trabalhos:
        trabalho = grupo.get('work-summary', [{}])[0]

        titulo = trabalho.get('title', {}).get('title', {}).get('value', pd.NA)
        tipo = (trabalho.get('type') or '').lower()

        pub_date = trabalho.get('publication-date') or {}
        ano_raw = pub_date.get('year', {}).get('value') if pub_date else None
        ano = pd.NA if not ano_raw else int(ano_raw)

        doi = extrair_doi(trabalho)
        autores = extrair_autores(trabalho)

        if tipo in TIPOS_CONGRESSO:
            # --- Trabalho de Congresso/Evento ---
            evento_dict = trabalho.get('journal-title') or {}
            titulo_evento = evento_dict.get('value', pd.NA)

            lista_artigos_congresso.append({
                'id_lattes': id_lattes,
                'titulo_artigo': titulo,
                'ano': ano,
                'doi': doi,
                'autores': autores,
                'titulo_evento_lattes': titulo_evento,
                'paginas': pd.NA,
                'sigla_evento_google': pd.NA,
                'titulo_evento_google': pd.NA,
                'estrato': pd.NA,
                'tipo_match': 'ORCID',
                'coautoria_aluno': pd.NA,
            })
        else:
            # --- Artigo de Periódico (default para os demais tipos) ---
            revista_dict = trabalho.get('journal-title') or {}
            revista = revista_dict.get('value', pd.NA)

            lista_artigos_periodico.append({
                'id_lattes': id_lattes,
                'titulo_artigo': titulo,
                'titulo_revista_lattes': revista,
                'ano_pub': ano,
                'doi': doi,
                'autores': autores,
                'match_adequado': pd.NA,
                'coautoria_aluno': pd.NA,
                'id_scopus': pd.NA,
                'titulo_revista_scopus': pd.NA,
                'maior_percentil': pd.NA,
                'codigo_area_maior_percentil': pd.NA,
                'area_maior_percentil': pd.NA,
                'issn': pd.NA,
                'computation_area': pd.NA,
            })

    # Respeita o rate-limit público da API do ORCID
    time.sleep(0.2)

print("\nExtração concluída.")
print(f"Artigos de periódico extraídos: {len(lista_artigos_periodico)}")
print(f"Trabalhos de congresso extraídos: {len(lista_artigos_congresso)}")

Processando ORCID 0000-0003-0057-7670 (id_lattes=0000000000000001)...
OI {'last-modified-date': {'value': 1784579706744}, 'educations': {'last-modified-date': {'value': 1580734859916}, 'education-summary': [{'created-date': {'value': 1471651850792}, 'last-modified-date': {'value': 1580734687597}, 'source': {'source-orcid': {'uri': 'http://orcid.org/0000-0003-0057-7670', 'path': '0000-0003-0057-7670', 'host': 'orcid.org'}, 'source-client-id': None, 'source-name': {'value': 'Pedro Henrique González'}}, 'department-name': 'Systems Engineering and Computers Science Program', 'role-title': 'Post Doctorate', 'start-date': {'year': {'value': '2015'}, 'month': {'value': '11'}, 'day': {'value': '01'}}, 'end-date': {'year': {'value': '2017'}, 'month': {'value': '02'}, 'day': {'value': '10'}}, 'organization': {'name': 'Federal University of Rio de Janeiro', 'address': {'city': 'Rio de Janeiro', 'region': '', 'country': 'BR'}, 'disambiguated-organization': {'disambiguated-organization-identifier':

## 4. Consolidação nos DataFrames finais (mesmo schema do `analyse.ipynb`)

As colunas abaixo replicam exatamente:

- `df_artigos_final` → `id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub, doi, autores, match_adequado, coautoria_aluno, id_scopus, titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil, area_maior_percentil, issn, computation_area`
- `df_artigos_congresso_final` → `id_lattes, titulo_artigo, ano, doi, autores, titulo_evento_lattes, paginas, sigla_evento_google, titulo_evento_google, estrato, tipo_match, coautoria_aluno`

Os campos que o ORCID não fornece diretamente (ex.: `maior_percentil`, `estrato`,
`match_adequado`) ficam como nulos — eles podem ser preenchidos depois pelo
mesmo processo de cruzamento com Scopus/base de eventos que o `analyse.ipynb`
já implementa, se desejar reaproveitar aquela lógica também para os dados do ORCID.

In [4]:
colunas_periodico = [
    'id_lattes', 'titulo_artigo', 'titulo_revista_lattes', 'ano_pub', 'doi',
    'autores', 'match_adequado', 'coautoria_aluno', 'id_scopus',
    'titulo_revista_scopus', 'maior_percentil', 'codigo_area_maior_percentil',
    'area_maior_percentil', 'issn', 'computation_area',
]

colunas_congresso = [
    'id_lattes', 'titulo_artigo', 'ano', 'doi', 'autores',
    'titulo_evento_lattes', 'paginas', 'sigla_evento_google',
    'titulo_evento_google', 'estrato', 'tipo_match', 'coautoria_aluno',
]

df_artigos_periodico_orcid = pd.DataFrame(lista_artigos_periodico, columns=colunas_periodico)
df_artigos_congresso_orcid = pd.DataFrame(lista_artigos_congresso, columns=colunas_congresso)

# Mesma tipagem usada no analyse.ipynb para permitir o concat sem surpresas
if not df_artigos_periodico_orcid.empty:
    df_artigos_periodico_orcid['ano_pub'] = pd.to_numeric(df_artigos_periodico_orcid['ano_pub'], errors='coerce').astype('Int64')
    df_artigos_periodico_orcid['id_lattes'] = df_artigos_periodico_orcid['id_lattes'].astype(str)

if not df_artigos_congresso_orcid.empty:
    df_artigos_congresso_orcid['ano'] = pd.to_numeric(df_artigos_congresso_orcid['ano'], errors='coerce').astype('Int64')
    df_artigos_congresso_orcid['id_lattes'] = df_artigos_congresso_orcid['id_lattes'].astype(str)

print("=== df_artigos_periodico_orcid ===")
display(df_artigos_periodico_orcid.head())
df_artigos_periodico_orcid.info()

print("\n=== df_artigos_congresso_orcid ===")
display(df_artigos_congresso_orcid.head())
df_artigos_congresso_orcid.info()

=== df_artigos_periodico_orcid ===


,id_lattes,titulo_artigo,titulo_revista_lattes,ano_pub,doi,autores,match_adequado,coautoria_aluno,id_scopus,titulo_revista_scopus,maior_percentil,codigo_area_maior_percentil,area_maior_percentil,issn,computation_area
0,0000000000000001,"Design, execution, and contextual factors shap...",<NA>,2026,10.1016/j.infsof.2026.108136,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,0000000000000001,A Critical Reflection on the State of Data Ana...,<NA>,2026,10.1145/3799715,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,0000000000000001,Exploring Technology Probe Applications in Sof...,<NA>,2026,10.2139/ssrn.6443058,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,0000000000000001,Experimental Evaluation of a Checklist-Based I...,<NA>,2025,10.1007/s10664-025-10681-7,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,0000000000000001,Testing Context-Aware Software Systems From th...,<NA>,2025,10.1109/TII.2025.3529918,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


<class 'pandas.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id_lattes                    69 non-null     str   
 1   titulo_artigo                69 non-null     str   
 2   titulo_revista_lattes        0 non-null      object
 3   ano_pub                      69 non-null     Int64 
 4   doi                          39 non-null     str   
 5   autores                      0 non-null      object
 6   match_adequado               0 non-null      object
 7   coautoria_aluno              0 non-null      object
 8   id_scopus                    0 non-null      object
 9   titulo_revista_scopus        0 non-null      object
 10  maior_percentil              0 non-null      object
 11  codigo_area_maior_percentil  0 non-null      object
 12  area_maior_percentil         0 non-null      object
 13  issn                         0 non-null      obj

,id_lattes,titulo_artigo,ano,doi,autores,titulo_evento_lattes,paginas,sigla_evento_google,titulo_evento_google,estrato,tipo_match,coautoria_aluno


<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id_lattes             0 non-null      object
 1   titulo_artigo         0 non-null      object
 2   ano                   0 non-null      object
 3   doi                   0 non-null      object
 4   autores               0 non-null      object
 5   titulo_evento_lattes  0 non-null      object
 6   paginas               0 non-null      object
 7   sigla_evento_google   0 non-null      object
 8   titulo_evento_google  0 non-null      object
 9   estrato               0 non-null      object
 10  tipo_match            0 non-null      object
 11  coautoria_aluno       0 non-null      object
dtypes: object(12)
memory usage: 132.0+ bytes


## 5. Próximo passo: unificar com `analyse.ipynb`

Depois de rodar este notebook e o `analyse.ipynb` (e o `Quick-Start.ipynb`,
adaptado para Scopus), basta concatenar os DataFrames que têm o mesmo schema:

```python
df_periodicos_unificado = pd.concat(
    [df_artigos_final, df_artigos_periodico_orcid, df_artigos_periodico_scopus],
    ignore_index=True,
)

df_congressos_unificado = pd.concat(
    [df_artigos_congresso_final, df_artigos_congresso_orcid],
    ignore_index=True,
)
```

E então subir para o DuckDB usando as mesmas tabelas `tb_artigo_periodico` e
`tb_artigo_conferencia` já criadas pelo `analyse.ipynb` (a FK `id_lattes`
referencia `tb_professores`, então garanta que toda pessoa já exista lá antes
de inserir).